# AI Customer Support Ticket Resolution Agent
**Portfolio project | Python | Retrieval-Augmented Generation (RAG concepts) | Ticket triage | Human escalation**

This notebook demonstrates a safe, beginner-friendly support workflow using a small synthetic knowledge base. It:
1. Classifies ticket category, priority, and sentiment using transparent baseline rules.
2. Retrieves relevant support articles with TF-IDF similarity.
3. Drafts a response grounded in the retrieved article.
4. Escalates sensitive, urgent, or low-confidence tickets to a human.
5. Runs simple checks against sample tickets.

> This is a portfolio prototype, not a production support system. It uses synthetic data and does not issue refunds, change orders, or perform other irreversible actions.

## 1. Install dependencies
Run this cell once in Jupyter or Google Colab. If scikit-learn is already installed, it will simply confirm the dependency is available.

In [ ]:
%pip -q install pandas scikit-learn

## 2. Imports and sample knowledge base
The knowledge base contains fictional support policies. Replace these with approved company documents for a real deployment.

In [ ]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

knowledge_base = [
    {
        "article_id": "KB-001",
        "title": "Password reset",
        "content": "To reset your password, open Sign In, choose Forgot Password, and follow the secure link sent to your registered email. Never share your password or verification code with support."
    },
    {
        "article_id": "KB-002",
        "title": "Order tracking",
        "content": "To track an order, open Orders in your account and select the order. Tracking updates may take 24 hours after dispatch. If tracking has not updated after 48 hours, contact support."
    },
    {
        "article_id": "KB-003",
        "title": "Refund policy",
        "content": "Refund eligibility depends on the product and order status. Support must verify the order and applicable policy before promising a refund. Requests involving disputed charges should be reviewed by a human agent."
    },
    {
        "article_id": "KB-004",
        "title": "Damaged item",
        "content": "If an item arrives damaged, keep the packaging and take clear photos of the item and parcel. Contact support with the order number and photos so the team can review replacement or refund options."
    },
    {
        "article_id": "KB-005",
        "title": "Cancel an order",
        "content": "Orders can only be cancelled while they are eligible for cancellation. Open Orders, select the order, and check whether Cancel Order is available. If the order has shipped, contact support for next steps."
    },
    {
        "article_id": "KB-006",
        "title": "Billing and duplicate charges",
        "content": "If you see a duplicate charge, do not share full card details. Provide the transaction date and order reference through the secure support channel. A human billing specialist should review disputed or duplicate charges."
    }
]

kb_df = pd.DataFrame(knowledge_base)
kb_df[["article_id", "title"]]

## 3. Build a lightweight retrieval index
TF-IDF is a simple baseline for a portfolio demo. For a more advanced version, replace it with embeddings and a vector database such as FAISS or Chroma.

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
kb_matrix = vectorizer.fit_transform(kb_df["title"] + " " + kb_df["content"])

def retrieve_articles(query, top_k=2):
    query_vec = vectorizer.transform([query])
    scores = cosine_similarity(query_vec, kb_matrix).ravel()
    ranked = scores.argsort()[::-1][:top_k]
    results = []
    for idx in ranked:
        row = kb_df.iloc[idx].to_dict()
        row["similarity"] = float(scores[idx])
        results.append(row)
    return results

retrieve_articles("I cannot log in and need to reset my password")

## 4. Ticket triage
This baseline uses explicit rules so that the notebook runs without an LLM API key. In a later iteration, use structured LLM output and evaluate it against a labeled dataset.

In [ ]:
CATEGORY_RULES = {
    "account_access": ["password", "login", "log in", "sign in", "locked out", "account"],
    "shipping": ["shipping", "tracking", "delivery", "late", "package", "parcel", "order status"],
    "refund": ["refund", "money back", "return my money"],
    "damaged_item": ["damaged", "broken", "cracked", "defective"],
    "cancellation": ["cancel", "cancellation"],
    "billing": ["charged", "charge", "billing", "duplicate", "payment", "card"],
}

URGENT_TERMS = ["urgent", "immediately", "fraud", "stolen", "security breach", "cannot access"]
NEGATIVE_TERMS = ["angry", "frustrated", "terrible", "unacceptable", "disappointed", "upset"]

def contains_any(text, terms):
    text = text.lower()
    return any(term in text for term in terms)

def classify_ticket(ticket_text):
    text = ticket_text.lower()
    category = "general"
    for candidate, terms in CATEGORY_RULES.items():
        if contains_any(text, terms):
            category = candidate
            break

    priority = "high" if contains_any(text, URGENT_TERMS) else "normal"
    sentiment = "negative" if contains_any(text, NEGATIVE_TERMS) else "neutral"

    # Sensitive categories should be reviewed by a person.
    needs_human = category in {"refund", "billing"} or priority == "high"
    return {
        "category": category,
        "priority": priority,
        "sentiment": sentiment,
        "needs_human": needs_human,
    }

classify_ticket("I am frustrated because my package tracking has not updated.")

## 5. Draft a grounded response
The response template uses retrieved article text rather than inventing policy details. A low retrieval score or sensitive issue triggers human review.

In [ ]:
def resolve_ticket(ticket_text, similarity_threshold=0.08):
    triage = classify_ticket(ticket_text)
    retrieved = retrieve_articles(ticket_text, top_k=2)
    best = retrieved[0] if retrieved else None

    if best is None or best["similarity"] < similarity_threshold:
        return {
            **triage,
            "status": "needs_human_review",
            "matched_article": None,
            "similarity": 0.0 if best is None else best["similarity"],
            "draft_reply": (
                "Thanks for contacting support. I could not confidently match your issue "
                "to an approved help article, so I’m routing this ticket to a human support agent."
            ),
            "sources": []
        }

    if triage["needs_human"]:
        status = "human_review_required"
        reply = (
            f"Thanks for contacting support. I found the relevant guidance in "
            f"{best['article_id']} ({best['title']}). Because this issue may require "
            "account-specific review, a human support specialist should verify the details "
            "before any commitment or action is made. " + best["content"]
        )
    else:
        status = "draft_ready"
        reply = (
            f"Thanks for contacting support. Based on our help article "
            f"{best['article_id']} ({best['title']}): {best['content']} "
            "If this does not resolve the issue, reply and we’ll route it to a support specialist."
        )

    return {
        **triage,
        "status": status,
        "matched_article": best["article_id"],
        "similarity": round(best["similarity"], 3),
        "draft_reply": reply,
        "sources": [{"article_id": x["article_id"], "title": x["title"], "similarity": round(x["similarity"], 3)} for x in retrieved]
    }

sample = resolve_ticket("My order tracking has not updated for three days and I am frustrated.")
sample

## 6. Try a few example tickets

In [ ]:
sample_tickets = [
    "I forgot my password and cannot sign in.",
    "My package tracking has not updated for three days.",
    "I see a duplicate charge on my card and need help.",
    "The item arrived cracked and damaged.",
    "URGENT: I think someone stole my account.",
    "I have a question that is not covered by the help articles."
]

results = [resolve_ticket(ticket) for ticket in sample_tickets]
results_df = pd.DataFrame([
    {
        "ticket": ticket,
        "category": result["category"],
        "priority": result["priority"],
        "sentiment": result["sentiment"],
        "status": result["status"],
        "matched_article": result["matched_article"],
        "similarity": result["similarity"]
    }
    for ticket, result in zip(sample_tickets, results)
])
results_df

## 7. Inspect one generated response

In [ ]:
ticket = "I see a duplicate charge on my card. Please help me."
result = resolve_ticket(ticket)

print("Ticket:", ticket)
print("Category:", result["category"])
print("Priority:", result["priority"])
print("Status:", result["status"])
print("Matched article:", result["matched_article"])
print("\nDraft reply:\n", result["draft_reply"])
print("\nSources:", result["sources"])

## 8. Basic validation checks
These checks verify core safety expectations for this small demo. They are not a substitute for a comprehensive evaluation suite.

In [ ]:
# Sensitive billing/refund cases must be routed for human review.
billing_case = resolve_ticket("I was charged twice for my order.")
assert billing_case["needs_human"] is True
assert billing_case["status"] == "human_review_required"

# Urgent security issues must be routed for human review.
security_case = resolve_ticket("URGENT: someone stole my account.")
assert security_case["priority"] == "high"
assert security_case["needs_human"] is True

# Password reset should retrieve the corresponding article.
password_case = resolve_ticket("How can I reset my password?")
assert password_case["matched_article"] == "KB-001"

print("All baseline validation checks passed.")

## 9. Suggested upgrades for your GitHub portfolio

- Replace keyword triage with structured LLM classification and a labeled evaluation set.
- Replace TF-IDF with embeddings and FAISS/Chroma vector retrieval.
- Use LangGraph to model triage → retrieval → draft → policy check → human approval.
- Add confidence thresholds, source citations, and a reviewer that blocks unsupported claims.
- Add FastAPI endpoints and a Streamlit interface.
- Add unit tests, CI with GitHub Actions, logging, and latency/cost tracking.
- Add authentication and role-based access before using real customer data.

### Important safety note
Keep this project on synthetic data until privacy, security, access control, policy review, and human escalation have been implemented. The prototype only drafts responses; it does not execute refunds or account changes.